<a href="https://colab.research.google.com/github/YAN-JINGHAO/TorchCode/blob/main/templates/07_batchnorm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/07_batchnorm.ipynb)

# 🟡 Medium: Implement BatchNorm

Implement **Batch Normalization** with both **training** and **inference** behavior.

In training mode, use **batch statistics** and update running estimates:

$$\text{BN}(x) = \gamma \cdot \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} + \beta$$

where $\mu_B$ and $\sigma_B^2$ are the mean and variance computed **across the batch** (dim=0).

In inference mode, use the provided **running mean/var** instead of current batch stats.

### Signature
```python
def my_batch_norm(
    x: torch.Tensor,
    gamma: torch.Tensor,
    beta: torch.Tensor,
    running_mean: torch.Tensor,
    running_var: torch.Tensor,
    eps: float = 1e-5,
    momentum: float = 0.1,
    training: bool = True,
) -> torch.Tensor:
    # x: (N, D) — normalize each feature across all samples in the batch
    # running_mean, running_var: updated in-place during training; used as-is during inference
```

### Rules
- Do **NOT** use `F.batch_norm`, `nn.BatchNorm1d`, etc.
- Compute batch mean and variance over `dim=0` with `unbiased=False`
- Update running stats like PyTorch: `running = (1 - momentum) * running + momentum * batch_stat`
- Use `running_mean` / `running_var` for inference when `training=False`
- Must support autograd w.r.t. `x`, `gamma`, `beta` (running statistics should be treated as buffers, not parameters requiring gradients)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.4 MB/s eta 0:00:00


In [2]:
import torch

In [43]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_batch_norm(
    x,
    gamma,
    beta,
    running_mean,
    running_var,
    eps=1e-5,
    momentum=0.1,
    training=True,
):
    if training:
      batch_mean = x.mean(dim=0)
      batch_var = x.var(dim=0, correction=0) # unbiased=False

      # Update running statistics in-place.
      # Detach to avoid tracking gradients. Or use torch.no_grad()
      running_mean.mul_(1 - momentum).add_(momentum * batch_mean.detach())
      running_var.mul_(1 - momentum).add_(momentum * batch_var.detach())

      mean = batch_mean
      var = batch_var
    else:
      mean = running_mean
      var = running_var

    x_norm = (x - mean) / torch.sqrt(var + eps)
    return gamma * x_norm + beta

In [39]:
x = torch.randn(8, 4)
print(x)
x_mean1 = x.mean(dim=0, keepdim=True)
x_mean2 = x.mean(dim=0)
print(x_mean1)
print(x_mean2)
print(x - x_mean1)
print(x - x_mean2)

tensor([[ 0.4386, -0.0107,  1.3384, -0.2794],
        [-0.5518, -2.8891, -1.5100,  1.0241],
        [ 0.1954, -0.7371,  1.7001,  0.3462],
        [ 0.9711,  1.4503, -0.0519, -0.6284],
        [-0.6538,  1.7198, -0.9610, -0.6375],
        [ 0.0747,  0.5600,  0.5314,  1.2351],
        [-1.1070, -1.7174,  1.5346, -0.0032],
        [-1.6034,  0.0581, -0.6302,  0.7466]])
tensor([[-0.2795, -0.1958,  0.2439,  0.2254]])
tensor([-0.2795, -0.1958,  0.2439,  0.2254])
tensor([[ 0.7182,  0.1851,  1.0944, -0.5048],
        [-0.2723, -2.6933, -1.7539,  0.7987],
        [ 0.4749, -0.5413,  1.4562,  0.1208],
        [ 1.2506,  1.6460, -0.2958, -0.8539],
        [-0.3743,  1.9156, -1.2049, -0.8629],
        [ 0.3542,  0.7557,  0.2875,  1.0097],
        [-0.8275, -1.5216,  1.2906, -0.2286],
        [-1.3239,  0.2539, -0.8742,  0.5212]])
tensor([[ 0.7182,  0.1851,  1.0944, -0.5048],
        [-0.2723, -2.6933, -1.7539,  0.7987],
        [ 0.4749, -0.5413,  1.4562,  0.1208],
        [ 1.2506,  1.6460, -0.29

In [44]:
# 🧪 Debug
x = torch.randn(8, 4)
gamma = torch.ones(4)
beta = torch.zeros(4)

# Running stats typically live on the same device and shape as features
running_mean = torch.zeros(4)
running_var = torch.ones(4)

# Training mode: uses batch stats and updates running_mean / running_var
out_train = my_batch_norm(x, gamma, beta, running_mean, running_var, training=True)
print("[Train] Output shape:", out_train.shape)
print("[Train] Column means:", out_train.mean(dim=0))   # should be ~0
print("[Train] Column stds: ", out_train.std(dim=0))    # should be ~1
print("Updated running_mean:", running_mean)
print("Updated running_var:", running_var)

# Inference mode: uses running_mean / running_var only
out_eval = my_batch_norm(x, gamma, beta, running_mean, running_var, training=False)
print("[Eval] Output shape:", out_eval.shape)

[Train] Output shape: torch.Size([8, 4])
[Train] Column means: tensor([-8.9407e-08,  7.4506e-09, -5.5879e-09, -1.4901e-08])
[Train] Column stds:  tensor([1.0690, 1.0690, 1.0690, 1.0690])
Updated running_mean: tensor([ 0.0642,  0.0594,  0.0314, -0.0385])
Updated running_var: tensor([0.9980, 0.9470, 1.0242, 0.9841])
[Eval] Output shape: torch.Size([8, 4])


In [45]:
# ✅ SUBMIT
from torch_judge import check
check("batchnorm")


🧪 Testing: Implement BatchNorm (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Training mode — zero mean per feature (3.1ms)
  ✅ [2/4] Training mode — numerical correctness and running stats update (4.0ms)
  ✅ [3/4] Inference mode — uses running statistics (1.8ms)
  ✅ [4/4] Gradient flow w.r.t inputs and affine params (1.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (10.2ms total)
  Progress saved. Run status() to see your dashboard.



In [15]:
from torch_judge import hint
hint("batchnorm")


💡 Hint for Implement BatchNorm:
   Implement train/eval BatchNorm: in training, use batch stats over dim=0 and update running_mean/running_var with momentum; in inference, normalize using the running statistics only.

